# Benchmark: how long one market step takes

How much time the market itself costs per environment step, before any
reinforcement learning: `LocalEnergyMarket.step` on SimBench LV networks of
different sizes, with one participant on every bus that has a load.

**What one timed step is.** Everything `LocalEnergyMarket.step` does
([lem.py](../../core/market/lem.py)): validate the orders, build them (choose
partners, if partner preference is on), the three clearing stages, settlement,
the power flow, reputation and the clock. The orders are **random** (buy or sell
at random, price uniform in `[min_price, max_price]`, quantity uniform in
`[0, 5]` kWh), as an untrained policy would send them.

**What is not in it.** Building the observation and the reward for the agents:
that layer does not exist yet in this branch, so an RL environment around the
market will cost a little more per step than measured here.

**Split of one step**, measured by wrapping the calls (nothing in `core/`
changes):

| Part | What it is |
|---|---|
| `power_flow` | `pandapower.runpp` alone |
| `grid_other` | the rest of `SimBenchGrid.assess`: setting every element's power for the step, the participants' injections, reading voltages and loadings |
| `market` | everything else: validation, orders, clearing, settlement, reputation |

Each network plays `DAYS` days; the first is a warm-up (pandapower and numba
compile on first use) and is left out of the averages. A day is 24 steps
(`STEP_HOURS = 1.0`) or 96 (`0.25`); the cost per step barely depends on it.

Results go to `notebooks/market/results/market_step.csv`, which the RL speed
benchmark reads (`notebooks/rl/benchmark_speed.ipynb`).

In [1]:
import sys
import time
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "core" / "market").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandapower
import pandas as pd

from core.config import placement
from core.config.scenario import GridSetup, MarketSetup, Scenario
from core.market.lem import LocalEnergyMarket, OrderIntent

pd.set_option("display.precision", 1)
pd.set_option("display.width", 160)

## a. Settings

- `NETWORKS`: SimBench LV networks, smallest to largest. With a participant on
  every load bus they give 13, 39, 53, 93 and 118 agents.
- `PARTNER_PREFERENCE`: run each network without and with partner preference
  (agents name a preferred partner, chosen by reputation and grid distance).
- `DAYS`: days played per network and setting; the first is a warm-up.

In [2]:
NETWORKS = [
    "1-LV-rural1--0-sw",
    "1-LV-semiurb4--0-sw",
    "1-LV-urban6--0-sw",
    "1-LV-rural2--0-sw",
    "1-LV-rural3--0-sw",
]
PARTNER_PREFERENCE = [False, True]
DAYS = 3
STEP_HOURS = 1.0
SEED = 0
MAX_QUANTITY_KWH = 5.0
RESULTS = ROOT / "notebooks" / "market" / "results" / "market_step.csv"

## b. The timing

`time_network` builds the scenario, plays `DAYS` days with random orders and
returns one row per step with the time of each part. The power flow is timed by
wrapping `pandapower.runpp`; the rest of the grid's work by wrapping the market's
`grid.assess` (a new grid object is built at every reset, so it is wrapped again
after each one).

In [3]:
class Clock:
    # Accumulates the time spent inside a wrapped function.
    def __init__(self):
        self.total = 0.0

    def wrap(self, fn):
        def timed(*args, **kwargs):
            start = time.perf_counter()
            try:
                return fn(*args, **kwargs)
            finally:
                self.total += time.perf_counter() - start

        return timed


def scenario_for(code: str, partner_preference: bool) -> Scenario:
    # One participant on every bus with a load (the network's own elements).
    scenario = Scenario(
        name=f"benchmark {code}",
        grid=GridSetup(code=code, step_hours=STEP_HOURS),
        market=MarketSetup(partner_preference=partner_preference),
    )
    kinds = [k for k in placement.kind_counts(scenario) if "load" in k]
    placement.place_all(scenario, kinds)
    return scenario


def random_intents(market: LocalEnergyMarket, rng) -> dict[str, OrderIntent]:
    m = market.scenario.market
    return {
        pid: OrderIntent(
            pid,
            is_buy=bool(rng.integers(2)),
            price=float(rng.uniform(m.min_price, m.max_price)),
            quantity_kwh=float(rng.uniform(0.0, MAX_QUANTITY_KWH)),
        )
        for pid in market.participants
    }


def time_network(code: str, partner_preference: bool) -> pd.DataFrame:
    scenario = scenario_for(code, partner_preference)
    rng = np.random.default_rng(SEED)
    runpp_clock = Clock()
    original_runpp = pandapower.runpp
    pandapower.runpp = runpp_clock.wrap(original_runpp)
    rows = []
    try:
        start = time.perf_counter()
        market = LocalEnergyMarket(scenario)
        build_s = time.perf_counter() - start
        for day in range(DAYS):
            start = time.perf_counter()
            market.reset()
            reset_s = time.perf_counter() - start
            assess_clock = Clock()
            market.grid.assess = assess_clock.wrap(market.grid.assess)
            while not market.done:
                intents = random_intents(market, rng)
                assess_clock.total = runpp_clock.total = 0.0
                start = time.perf_counter()
                market.step(intents)
                total = time.perf_counter() - start
                rows.append({
                    "network": code,
                    "agents": len(market.participants),
                    "partner_preference": partner_preference,
                    "day": day,
                    "step": market.current_step - 1,
                    "total_ms": 1000 * total,
                    "power_flow_ms": 1000 * runpp_clock.total,
                    "grid_other_ms": 1000 * (assess_clock.total - runpp_clock.total),
                    "market_ms": 1000 * (total - assess_clock.total),
                    "reset_s": reset_s,
                    "build_s": build_s,
                })
    finally:
        pandapower.runpp = original_runpp
    return pd.DataFrame(rows)

## c. Run

About a second per day on the small networks, a few seconds on the large ones.

In [4]:
frames = []
for code in NETWORKS:
    for pref in PARTNER_PREFERENCE:
        start = time.time()
        frame = time_network(code, pref)
        frames.append(frame)
        print(f"{code:22s} partner preference {pref!s:5s}: "
              f"{frame.agents.iloc[0]:3d} agents, {time.time() - start:5.1f} s")
steps = pd.concat(frames, ignore_index=True)

1-LV-rural1--0-sw      partner preference False:  13 agents,   8.2 s

1-LV-rural1--0-sw      partner preference True :  13 agents,   2.5 s

1-LV-semiurb4--0-sw    partner preference False:  39 agents,   6.2 s

1-LV-semiurb4--0-sw    partner preference True :  39 agents,   2.9 s

1-LV-urban6--0-sw      partner preference False:  53 agents,   6.1 s

1-LV-urban6--0-sw      partner preference True :  53 agents,   3.3 s

1-LV-rural2--0-sw      partner preference False:  93 agents,   7.2 s

1-LV-rural2--0-sw      partner preference True :  93 agents,   4.4 s

1-LV-rural3--0-sw      partner preference False: 118 agents,   7.7 s

1-LV-rural3--0-sw      partner preference True : 118 agents,   5.2 s

## d. Time per step

Mean over every day after the warm-up, in milliseconds. `first_step_ms` is the
first step of a day (it computes the grid paths and zones the day reuses),
`reset_s` the start of a new episode. `market_h_per_1M_steps` is how long one
million environment steps would spend in the market alone: the floor under any
training run on that network, before the learner adds anything.

In [5]:
timed = steps[steps.day > 0]
keys = ["network", "agents", "partner_preference"]
summary = timed.groupby(keys, sort=False).agg(
    total_ms=("total_ms", "mean"),
    market_ms=("market_ms", "mean"),
    grid_other_ms=("grid_other_ms", "mean"),
    power_flow_ms=("power_flow_ms", "mean"),
    reset_s=("reset_s", "mean"),
)
summary["first_step_ms"] = (
    timed[timed.step == 0].groupby(keys, sort=False).total_ms.mean()
)
summary["power_flow_share"] = summary.power_flow_ms / summary.total_ms
summary["market_h_per_1M_steps"] = summary.total_ms * 1e6 / 1000 / 3600
summary = summary.reset_index()
summary

,network,agents,partner_preference,total_ms,market_ms,grid_other_ms,power_flow_ms,reset_s,first_step_ms,power_flow_share,market_h_per_1M_steps
0,1-LV-rural1--0-sw,13,False,29.5,0.5,6.1,22.9,2.0e-02,54.2,0.8,8.2
1,1-LV-rural1--0-sw,13,True,35.2,0.6,7.2,27.3,2.1e-02,58.1,0.8,9.8
2,1-LV-semiurb4--0-sw,39,False,40.1,1.5,14.1,24.4,3.5e-02,102.4,0.6,11.1
3,1-LV-semiurb4--0-sw,39,True,37.3,1.6,12.1,23.6,3.3e-02,102.4,0.6,10.4
4,1-LV-urban6--0-sw,53,False,40.9,2.1,15.0,23.8,6.6e-02,128.8,0.6,11.4
5,1-LV-urban6--0-sw,53,True,43.4,2.2,15.4,25.8,6.9e-02,129.2,0.6,12.1
6,1-LV-rural2--0-sw,93,False,59.0,7.3,26.1,25.6,7.7e-02,266.9,0.4,16.4
7,1-LV-rural2--0-sw,93,True,59.2,5.6,26.8,26.8,7.2e-02,223.5,0.5,16.5
8,1-LV-rural3--0-sw,118,False,61.4,7.2,30.0,24.2,9.1e-02,260.1,0.4,17.0
9,1-LV-rural3--0-sw,118,True,64.6,7.3,31.0,26.4,8.9e-02,319.5,0.4,17.9


## e. Save for the RL benchmark

In [6]:
RESULTS.parent.mkdir(parents=True, exist_ok=True)
summary.assign(step_hours=STEP_HOURS, days_timed=DAYS - 1).to_csv(RESULTS, index=False)
print("saved", RESULTS.relative_to(ROOT))

saved

notebooks\results\market_step.csv

## f. How to read it

- **The floor of a training run.** Multiply `total_ms` by the environment steps a
  run needs: 1 million steps on a network with `total_ms` = 40 is 11 hours of
  market alone, with the learner's time on top.
- **Where the time goes.** If `power_flow` and `grid_other` dominate, the market
  logic is not the problem; a faster power flow (e.g. the `lightsim2grid`
  backend, or skipping it during training), or several environments sampled in
  parallel processes, would be.
- **Sampling in parallel.** RLlib can step several copies of the environment at
  once (`num_env_runners`). With 6 cores, 4 runners would divide this floor by up
  to 4, if memory allows.